In [1]:
# Kill all processess on GPU
!fuser -v /dev/nvidia* -k

In [2]:
# Check GPU status
!nvidia-smi

Tue Jun 23 16:08:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Libraries

In [3]:
%%capture
!uv pip install evaluate

In [4]:
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForQuestionAnswering, 
    DataCollatorWithPadding, 
    Trainer, 
    TrainingArguments,
)
from datasets import load_dataset, Dataset
import evaluate

# Utilities

In [5]:
def load_test_dataset(
    lang, # e.g., 'en' | 'ja' | 'id'
    size, 
):
    # Set up Hugging Face dataset configuration
    data_id = 'google/xquad'
    data_dir = f'xquad.{lang}'
    split = 'validation'

    dataset_stream = load_dataset(
        data_id,
        data_dir=data_dir,
        split=split,
        streaming=True,
    )

    test_data = []

    for i, example in enumerate(dataset_stream):
        if i < size:
            test_data.append(example)
        else:
            break

    return Dataset.from_list(test_data)

# Configurations

In [6]:
# Run configuration
SEED = 42
LANG = 'vi'  # e.g., 'en' | 'ja' | 'id'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Model configuration
MODEL_ID = 'alxxtexxr/xlm-roberta-base-squad-en-LoRA-Merged-v260623145250'

# Data configuration
TEST_SIZE = 125

# Evaluation configuration
BATCH_SIZE = 16
EVAL_DIR = './eval/xquad_xlmr'

# Model

In [7]:
# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_ID)
model.eval().to(DEVICE)

print("device:", model.device)

device: cuda:0


# Data

In [8]:
# Preprocess the test dataset for evaluation
def preprocess_squad(examples):
    tokenized = tokenizer(
        examples['question'],
        examples['context'],
        truncation='only_second',
        max_length=384,
        return_offsets_mapping=True,
    )
    start_positions = []
    end_positions = []
    for i, offsets in enumerate(tokenized['offset_mapping']):
        sequence_ids = tokenized.sequence_ids(i)
        answer = examples['answers'][i]
        answer_start_char = answer['answer_start'][0]
        answer_text = answer['text'][0]
        answer_end_char = answer_start_char + len(answer_text)

        token_start = None
        token_end = None
        for idx, (offset_start, offset_end) in enumerate(offsets):
            if sequence_ids[idx] != 1:
                continue
            if offset_start >= answer_start_char and offset_end <= answer_end_char:
                if token_start is None:
                    token_start = idx
                token_end = idx
            elif offset_start < answer_end_char and offset_end > answer_start_char:
                if token_start is None:
                    token_start = idx
                token_end = idx

        if token_start is None or token_end is None:
            token_start = 0
            token_end = 0

        start_positions.append(token_start)
        end_positions.append(token_end)

    tokenized['start_positions'] = start_positions
    tokenized['end_positions'] = end_positions
    return tokenized

# Load the raw test dataset (with answers)
raw_dataset = load_test_dataset(LANG, size=TEST_SIZE)   # or load_dataset("xquad", ...)

# Tokenize it (this will remove the original columns like 'answers')
test_dataset = raw_dataset.map(preprocess_squad, batched=True, remove_columns=raw_dataset.column_names)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Map:   0%|          | 0/125 [00:00<?, ? examples/s]

# Evaluation

In [9]:
squad_metric = evaluate.load('squad')

def compute_metrics(pred):
    start_logits, end_logits = pred.predictions
    
    predictions = []
    for i, (start_log, end_log) in enumerate(zip(start_logits, end_logits)):
        start_idx = start_log.argmax()
        end_idx = end_log.argmax()
        input_ids = test_dataset[i]['input_ids']
        answer_ids = input_ids[start_idx : end_idx + 1]
        answer_text = tokenizer.decode(answer_ids, skip_special_tokens=True)
        predictions.append({
            'id': raw_dataset[i]['id'],
            'prediction_text': answer_text,
        })
    
    references = []
    for example in raw_dataset:
        references.append({
            'id': example['id'],
            'answers': {
                'text': example['answers']['text'],
                'answer_start': example['answers']['answer_start'],
            },
        })
    
    return squad_metric.compute(predictions=predictions, references=references)

In [10]:
# Set up the trainer for evaluation
data_collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)
eval_args = TrainingArguments(
    output_dir=EVAL_DIR,
    do_train=False,
    do_eval=True,
    per_device_eval_batch_size=BATCH_SIZE,
    dataloader_drop_last=False,
    report_to=[],
)
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=eval_args,
    compute_metrics=compute_metrics,
)
trainer.label_names = ['start_positions', 'end_positions']

In [14]:
# Run evaluation
predictions = trainer.predict(test_dataset)

# Print evaluation results
print("-" * 64)
print("EVALUATION RESULTS")
print("-" * 64)
for metric in predictions.metrics:
    print(f"{metric}: {predictions.metrics[metric]}")

----------------------------------------------------------------
EVALUATION RESULTS
----------------------------------------------------------------
test_loss: 1.9763044118881226
test_model_preparation_time: 0.0027
test_exact_match: 36.8
test_f1: 49.39117861391544
test_runtime: 2.3618
test_samples_per_second: 52.925
test_steps_per_second: 3.387


In [13]:
# Save predictions
predictions_path = f'{EVAL_DIR}/predictions.txt'
start_logits, end_logits = predictions.predictions

with open(predictions_path, 'w', encoding='utf-8') as f:
    for i in range(len(test_dataset)):
        start_idx = start_logits[i].argmax()
        end_idx = end_logits[i].argmax()
        input_ids = test_dataset[i]['input_ids']
        answer_ids = input_ids[start_idx:end_idx + 1]
        pred_text = tokenizer.decode(answer_ids, skip_special_tokens=True)
        gt_text = raw_dataset[i]['answers']['text'][0]
        
        f.write(f"Q: {raw_dataset[i]['question']}\n")
        f.write(f"GT: {gt_text}\n")
        f.write(f"Pred: {pred_text}\n")
        f.write(f"Match: {pred_text.strip().lower() == gt_text.strip().lower()}\n")
        f.write("-" * 64 + "\n")

print(f"Saved predictions to: {predictions_path}")

Saved predictions to: ./eval/xquad_xlmr/predictions.txt


# Multi-Language Evaluation

In [ ]:
# languages = ['en', 'ar', 'de', 'el', 'es', 'hi', 'ru', 'th', 'tr', 'vi', 'zh']
# results = {}

# for lang in languages:
#     dataset = load_dataset('xquad', f'xquad.{lang}', split='test')
#     test_dataset = dataset.map(preprocess_squad, batched=True, remove_columns=dataset.column_names)
#     predictions = trainer.predict(test_dataset)
#     metrics = compute_metrics(predictions)
#     results[lang] = metrics
#     print(f"{lang}: {metrics}")